In [1]:
import torch
import torch.nn as nn

class ConvNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ConvNet, self).__init__()
        
        # Input shape: (Batch, 3 Channels, 32 Height, 32 Width) - e.g., CIFAR-10 Images
        self.features = nn.Sequential(
            # Conv Layer 1: 3 input channels to 16 output feature maps
            # Output size formula: ((Input - Kernel + 2*Padding) / Stride) + 1
            # ((32 - 3 + 2*1) / 1) + 1 = 32 -> Shape: (Batch, 16, 32, 32)
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            # Max Pool 1: 2x2 window reduces spatial size by half
            # Shape transitions to: (Batch, 16, 16, 16)
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Conv Layer 2: 16 channels to 32 feature maps
            # ((16 - 3 + 2*1) / 1) + 1 = 16 -> Shape: (Batch, 32, 16, 16)
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            # Max Pool 2: Shape transitions to: (Batch, 32, 8, 8)
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # Classification Linear Head
        # Flattening 32 channels of 8x8 matrices = 32 * 8 * 8 = 2048 nodes
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(p=0.4),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        # Extract spatial image feature maps
        x = self.features(x)
        
        # Flatten the 3D tensor layers into a 1D vector per sample for the linear layers
        # x.size(0) preserves the batch dimension cleanly
        x = x.view(x.size(0), -1)
        
        # Compute final classification logits
        logits = self.classifier(x)
        return logits

# Instantiate the vision module
cnn_model = ConvNet(num_classes=10)
print(cnn_model)

# Verify tensor dimension paths with a dummy image batch (5 images, 3 channels, 32x32 pixels)
mock_images = torch.randn(5, 3, 32, 32)
output_logits = cnn_model(mock_images)
print(f"\nForward pass successful! Output tensor shape: {output_logits.shape} (Batch, Classes)")

ConvNet(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=2048, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)

Forward pass successful! Output tensor shape: torch.Size([5, 10]) (Batch, Classes)
